In [1]:
# Install openrouteservice for first time users
# !pip install openrouteservice

In [2]:
import openrouteservice
import pandas as pd
import numpy as np
from tqdm import tqdm

In [3]:
# Load in rental data
rental = pd.read_csv("rea_data/domain/Data/vic_rentals_all_cleaned.csv")
rental_coords = rental[["listing_id", "lon","lat"]]
rental_coords.head()

,listing_id,lon,lat
0,16782629,144.87091,-37.830982
1,17471867,144.86755,-37.826218
2,17721851,144.86632,-37.831226
3,17725855,144.86768,-37.827423
4,17745057,144.86790,-37.826270


In [4]:
# Load in bus stop data
bus = pd.read_csv("data\metro_bus_stops.txt")
bus_coords = bus[["stop_id", "stop_lon","stop_lat"]]
bus_coords.head()

,stop_id,stop_lon,stop_lat
0,1000,145.018951,-37.700775
1,10001,144.776152,-37.726975
2,10002,144.595789,-37.676159
3,10009,144.775899,-37.741497
4,1001,145.019685,-37.699183


In [5]:
# Define Haversine distance function
def haversine_vec(lon1, lat1, lon2, lat2):
    """
    Compute distance (meters) between two points using vectorised Haversine formula
    """
    lon1_rad = np.radians(lon1)[:, np.newaxis]
    lat1_rad = np.radians(lat1)[:, np.newaxis]

    lon2_rad = np.radians(lon2)[np.newaxis, :]
    lat2_rad = np.radians(lat2)[np.newaxis, :]

    earth_mean_radius = 6371000
    phi1, phi2 = lat1_rad, lat2_rad
    dphi1 = lat2_rad - lat1_rad
    dlambda = lon2_rad - lon1_rad
    
    a = np.sin(dphi1/2)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))

    return earth_mean_radius * c

In [6]:
# Find top 5 closest bus stops for each rental
dist_matrix = haversine_vec(rental_coords["lon"].values, rental_coords["lat"].values, bus_coords["stop_lon"].values, bus_coords["stop_lat"].values)

top5_indices = np.argsort(dist_matrix, axis=1)[:,:5]

rental_coords["top5_idx"] = list(top5_indices)
rental_coords["top5_tuple"] = [tuple(int(i) for i in sorted(x)) for x in top5_indices]

C:\Users\chinj\AppData\Local\Temp\ipykernel_19532\1667537328.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top5_idx"] = list(top5_indices)
C:\Users\chinj\AppData\Local\Temp\ipykernel_19532\1667537328.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  rental_coords["top5_tuple"] = [tuple(int(i) for i in sorted(x)) for x in top5_indices]


In [7]:
# Group results by their top 5 closest bus stops
groups = rental_coords.groupby("top5_tuple")["listing_id"].apply(list).reset_index()
groups["num_rentals"] = groups["listing_id"].apply(len)

In [ ]:
# API key for OpenRouteService (ORS); hidden here for privacy
API_KEY = NA # Replace NA with your actual key when running

# Initialize ORS client with the API key
client = openrouteservice.Client(key = API_KEY)

In [9]:
# Batch rentals to stay under API route limit
top_n_bus = 5
max_route_per_batch = 3500

batches = []
current_batch = []
current_batch_rentals = 0

# Loop over each group of rentals that share the same top 5 bus stops
for _, row in groups.iterrows():
    group_name = row["top5_tuple"]
    rentals = row["listing_id"]
    group_size = len(rentals)

    start = 0
    # Split group into chunks so that adding them won't exceed ORS route limit
    while start < group_size:
        # Estimate number of rentals allowed to add to current batch without exceeding ORS route limit
        num_groups_if_added = len(current_batch) + 1
        max_rentals_for_this_chunk = max(
            (max_route_per_batch // (num_groups_if_added * top_n_bus)) - current_batch_rentals,
            1
        )

        # Determine slice of rentals for this chunk
        end = min(start + max_rentals_for_this_chunk, group_size)
        chunk = rentals[start:end]

        # Compute estimated number of routes if this chunk is added
        num_origins = current_batch_rentals + len(chunk)
        num_destinations = num_groups_if_added * top_n_bus
        estimated_routes = num_origins * num_destinations

        # Start new batch if adding this chunk exceeds ORS route limit
        if estimated_routes >= max_route_per_batch: 
            if current_batch:
                batches.append(current_batch)
            current_batch = [(group_name, chunk)]     
            current_batch_rentals = len(chunk)
        else:
            # Add chunk to batch otherwise
            current_batch.append((group_name, chunk))
            current_batch_rentals += len(chunk)
        start = end

# Append last batch if there are any remaining rentals
if current_batch:
        batches.append(current_batch)

print(f"Created {len(batches)} batches with full ORS route limit logic.")

Created 308 batches with full ORS route limit logic.


In [10]:
# Print and check summary of batches
for i, batch in enumerate(batches):
    num_groups = len(batch)
    num_rentals = sum(len(rentals) for _, rentals in batch)
    print(f"Batch {i+1}: {num_groups} groups, {num_rentals} rentals")

Batch 1: 25 groups, 26 rentals
Batch 2: 21 groups, 33 rentals
Batch 3: 19 groups, 32 rentals
Batch 4: 22 groups, 30 rentals
Batch 5: 22 groups, 30 rentals
Batch 6: 25 groups, 26 rentals
Batch 7: 24 groups, 28 rentals
Batch 8: 24 groups, 29 rentals
Batch 9: 21 groups, 31 rentals
Batch 10: 21 groups, 32 rentals
Batch 11: 21 groups, 32 rentals
Batch 12: 24 groups, 27 rentals
Batch 13: 23 groups, 29 rentals
Batch 14: 24 groups, 27 rentals
Batch 15: 20 groups, 33 rentals
Batch 16: 25 groups, 27 rentals
Batch 17: 21 groups, 33 rentals
Batch 18: 22 groups, 30 rentals
Batch 19: 22 groups, 30 rentals
Batch 20: 23 groups, 29 rentals
Batch 21: 25 groups, 27 rentals
Batch 22: 24 groups, 29 rentals
Batch 23: 21 groups, 31 rentals
Batch 24: 24 groups, 28 rentals
Batch 25: 24 groups, 29 rentals
Batch 26: 15 groups, 43 rentals
Batch 27: 23 groups, 29 rentals
Batch 28: 20 groups, 34 rentals
Batch 29: 23 groups, 30 rentals
Batch 30: 22 groups, 31 rentals
Batch 31: 19 groups, 34 rentals
Batch 32: 15 grou

In [13]:
# Call ORS to get distance to closest bus stops
all_rental_ids = []
all_distances = []
all_bus_ids = []

for batch in tqdm(batches, desc="Processing batches"):
    batch_rentals = [r for _, rentals in batch for r in rentals]
    batch_coords = rental_coords.set_index("listing_id").loc[batch_rentals][["lon", "lat"]].values.tolist()

    batch_bus_indices = [
        rental_coords.loc[rental_coords["listing_id"]==r, "top5_idx"].values[0] for r in batch_rentals
    ]

    unique_bus_indices = list(set(idx for indices in batch_bus_indices for idx in indices))

    origins = batch_coords
    destinations = bus_coords.loc[bus_coords.index.isin(unique_bus_indices)][["stop_lon", "stop_lat"]].values.tolist()
    dest_ids_map = {idx: bus_coords["stop_id"].iloc[idx] for idx in unique_bus_indices}
   
    bus_pos_map = {idx: i for i, idx in enumerate(unique_bus_indices)}

    num_origins = len(origins)
    num_destinations = len(destinations)
    num_routes = num_origins * num_destinations
    print(f"Checking batch → {num_origins} origins × {num_destinations} destinations = {num_routes} routes")

    # Call ORS distance matrix
    matrix = client.distance_matrix(
        locations = origins + destinations,
        profile = "driving-car",
        metrics = ["distance"],
        sources = list(range(len(origins))),
        destinations = list(range(len(origins), len(origins)+len(destinations)))
    )

    # Extract distances and bus stop IDs for top 5 bus stops
    for i, rental in enumerate(batch_rentals):
        top_indices = batch_bus_indices[i]
        dest_positions = [bus_pos_map[idx] for idx in top_indices]
        rental_distances = [matrix["distances"][i][pos] for pos in dest_positions]
        rental_bus_ids = [dest_ids_map[idx] for idx in top_indices]

        cleaned_distances = [d if d is not None else float("inf") for d in rental_distances]

        sorted_idx = np.argsort(cleaned_distances)
        top_n_distances = [cleaned_distances[k] for k in sorted_idx]
        top_n_ids = [rental_bus_ids[k] for k in sorted_idx]

        all_rental_ids.append(batch_rentals[i])
        all_distances.append(top_n_distances)
        all_bus_ids.append(top_n_ids)

Processing batches:   0%|          | 0/308 [00:00<?, ?it/s]

Checking batch → 26 origins × 77 destinations = 2002 routes


Processing batches:   0%|          | 1/308 [00:02<15:14,  2.98s/it]

Checking batch → 33 origins × 62 destinations = 2046 routes


Processing batches:   1%|          | 2/308 [00:03<08:24,  1.65s/it]

...

Processing batches:  99%|█████████▉| 306/308 [12:37<00:00,  2.04it/s]

Checking batch → 34 origins × 47 destinations = 1598 routes


Processing batches: 100%|█████████▉| 307/308 [12:38<00:00,  2.14it/s]

Checking batch → 27 origins × 70 destinations = 1890 routes


Processing batches: 100%|██████████| 308/308 [12:38<00:00,  2.46s/it]


In [14]:
# Save ORS distances for bus stops into dataframe
df = pd.DataFrame({
    "rental_id": all_rental_ids,
    "top_distances": all_distances,
    "top_bus_ids": all_bus_ids
})

df.head()

,rental_id,top_distances,top_bus_ids
0,17747586,"[433.52, 581.43, 14211.31, 25808.42, 55651.27]","[1001, 1000, 10345, 1201, 10837]"
1,17292142,"[440.8, 536.4, 960.47, 26116.62, 26390.48]","[1000, 1001, 1202, 1201, 1200]"
2,17717715,"[671.15, 2716.4, 25618.05, 26167.44, 53286.2]","[1000, 998, 1199, 999, 1198]"
3,17721816,"[765.37, 26715.05, 27772.45, 27990.99, 29923.31]","[1000, 1200, 22361, 999, 22359]"
4,16512589,"[803.2, 21361.67, 21659.3, 28089.64, 28258.79]","[10001, 22678, 22674, 10308, 10307]"


In [15]:
# Save final CSV
df.to_csv("rentals_distances_to_bus_stops.csv", index=False)